In [13]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns 

In [14]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source'],
      dtype='str')

In [15]:
all_data_systems = systems_cleaned[
    systems_cleaned['has_current_data']
    & systems_cleaned['has_irradiance_data']
    & systems_cleaned['has_power_data']
    & systems_cleaned['has_temperature_data']
    & systems_cleaned['has_voltage_data']
    & systems_cleaned['has_ambient_temperature_data']
]
all_data_ids = set(all_data_systems.system_id)

In [16]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [17]:
my_system_ids = list(all_data_ids.intersection(metrics_id_set))
my_system_ids.sort()
num_ids = len(my_system_ids)
num_ids

36

In [156]:
system_id = 1207
relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
for j in relevant_rows_systems.index[0:1]:
    max_dc_capacity = relevant_rows_systems.loc[j, 'dc_capacity_kW']
    system_type = relevant_rows_systems.loc[j, 'simplified_type']
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]

In [157]:
relevant_rows_metrics

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
208,1207,2622,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__2622
209,1207,2633,poa_irradiance,Irradiance POA,W/m^2,W/m^2,1.0,0.0,,avg,NaN,NaN,,poa_irradiance__2633
210,1207,2629,module_temp,Temperature module,C,C,1.0,0.0,,avg,NaN,NaN,,module_temp__2629
211,1207,2623,dc_power_A1,DC power,-,-,1.0,0.0,,NaN,NaN,NaN,,dc_power_a1__2623
212,1207,2624,dc_power_A2,DC power,-,-,1.0,0.0,,NaN,NaN,NaN,,dc_power_a2__2624
213,1207,2625,dc_power_B1,DC power,-,-,1.0,0.0,,NaN,NaN,NaN,,dc_power_b1__2625
214,1207,2626,dc_power_B2,DC power,-,-,1.0,0.0,,NaN,NaN,NaN,,dc_power_b2__2626
215,1207,2627,dc_power_C1,DC power,-,-,1.0,0.0,,NaN,NaN,NaN,,dc_power_c1__2627
216,1207,2628,dc_power_C2,DC power,-,-,1.0,0.0,,NaN,NaN,NaN,,dc_power_c2__2628
217,1207,2630,poa_irradiance_cs300_1,Irradiance POA,W/m^2,W/m^2,1.0,0.0,,NaN,NaN,NaN,,poa_irradiance_cs300_1__2630


In [43]:
def metrics_search_for_fragment_df(df: pd.DataFrame, fragment: str):
    fragment = fragment.lower()
    return df[
        (df.loc[:, 'sensor_name'].str.contains(fragment, case=False))
        | (df.loc[:, 'common_name'].str.contains(fragment, case=False))
    ]

In [54]:
def metrics_search_for_two_fragments_df(df: pd.DataFrame, fragment_1: str,
                                        fragment_2: str, and_or: str):
    fragment_1 = fragment_1.lower()
    fragment_2 = fragment_2.lower()
    if and_or == 'and':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False)))
            & ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False)))
        ]
    elif and_or == 'or':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False)))
            | ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False)))
        ]

In [57]:
# sample use -- search for dc and power
system_id = 1283
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
metrics_search_for_two_fragments_df(relevant_rows_metrics, 'pow', 'dc', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
502,1283,1134,inv1_dc_power,DC power,W,W,1.0,0.0,inv1_dc_voltage*inv1_dc_current,avg,NaN,NaN,,inv1_dc_power__1134
503,1283,1135,inv2_dc_power,DC power,W,W,1.0,0.0,inv2_dc_voltage*inv2_dc_current,avg,NaN,NaN,,inv2_dc_power__1135
504,1283,1136,dc_power,DC power,W,W,1.0,0.0,inv1_dc_power+inv2_dc_power,avg,NaN,NaN,,dc_power__1136


In [26]:
def construct_all_parquet_power_aggregator_names():
    global metrics_df, systems_cleaned
    power_data = metrics_search_for_fragment_df(metrics_df, 'pow')
    power_ids = set(power_data.system_id).intersection(set(systems_cleaned.system_id))
    answers_dict = {
        id: [] for id in power_ids
    }
    # manually computed list
    dc_sensor_names = ['dc_power', 'dc_power_hW', 'dc_power_kW', 'dc_power_1_6',
                       'InvPDC_kW_Avg', 'dc_power_calc']
    ac_sensor_names = ['ac_power', 'ac_power_hW', 'ac_power_kW', 'ac_power_1_6',
                       'InvPAC_kW_Avg', 'ac_power_calc', 'W_avg',
                       'ac_power_metered_kW', 'RTW']
    for sensor_name in dc_sensor_names + ac_sensor_names:
        exact_name_metrics = metrics_df[
                metrics_df['sensor_name'] == sensor_name
        ]
        for id in set(exact_name_metrics['system_id']).intersection(set(systems_cleaned.system_id)):
            relevant_rows_metrics = metrics_df[(metrics_df['system_id']==id)
                                               & (metrics_df['sensor_name']==sensor_name)]
            if len(relevant_rows_metrics.index) > 1:
                raise RuntimeError(f'System {id} has multiple sensors named {sensor_name}!')
            else:
                ind = relevant_rows_metrics.index[0]
                metric_id = relevant_rows_metrics.loc[ind, 'metric_id']
                common_name = relevant_rows_metrics.loc[ind, 'common_name']
                given_unit = relevant_rows_metrics.loc[ind, 'units']
                calc_type = relevant_rows_metrics.loc[ind, 'calc_details']
                answers_dict[id].append({
                    'metric_id': metric_id,
                    'sensor_name': sensor_name,
                    'common_name': common_name,
                    'units': given_unit,
                    'calc_details': calc_type
                })
    # quick checks
    for id in power_ids:
        # check for missing entries
        if len(answers_dict[id]) == 0:
            print(f'System {id} appears to have no obvious power aggregator name.')

    return answers_dict

Justify the inclusion of 'W_avg', 'RTW' in ac sensor names!

In [22]:
# show nothing but AC power
assert(set(metrics_df[metrics_df['sensor_name'] == 'W_avg'].common_name) == {'AC power'})
assert(set(metrics_df[metrics_df['sensor_name'] == 'RTW'].common_name) == {'AC power'})

In [27]:
agg_power = construct_all_parquet_power_aggregator_names()

In [24]:
def sort_all_dc_power_names_step_1():
    global metrics_df, systems_cleaned
    power_data = metrics_search_for_fragment_df(metrics_df, 'pow')
    dc_data = metrics_search_for_fragment_df(metrics_df, 'dc')
    power_ids = set(power_data.system_id).intersection(set(systems_cleaned.system_id),
                                                       set(dc_data.system_id))
    # remove 3 false positives -- systems with DC voltage but not DC power
    power_ids = power_ids.difference({1367, 1432, 1433})
    answers_dict = {
        id: [] for id in power_ids
    }
    dc_agg_sensor_names = ['dc_power', 'dc_power_hW', 'dc_power_kW', 'dc_power_1_6',
                       'InvPDC_kW_Avg', 'dc_power_calc']
    for sensor_name in dc_agg_sensor_names:
        exact_name_metrics = metrics_df[
                metrics_df['sensor_name'] == sensor_name
        ]
        for id in set(exact_name_metrics['system_id']).intersection(set(systems_cleaned.system_id)):
            relevant_rows_metrics = metrics_df[(metrics_df['system_id']==id)
                                               & (metrics_df['sensor_name']==sensor_name)]
            if len(relevant_rows_metrics.index) > 1:
                raise RuntimeError(f'System {id} has multiple sensors named {sensor_name}!')
            else:
                ind = relevant_rows_metrics.index[0]
                metric_id = relevant_rows_metrics.loc[ind, 'metric_id']
                common_name = relevant_rows_metrics.loc[ind, 'common_name']
                given_unit = relevant_rows_metrics.loc[ind, 'units']
                calc_type = relevant_rows_metrics.loc[ind, 'calc_details']
                answers_dict[id].append({
                    'metric_id': metric_id,
                    'sensor_name': sensor_name,
                    'common_name': common_name,
                    'units': given_unit,
                    'calc_details': calc_type,
                    'whole_or_part': 'whole'
                })
    # quick checks
    for id in power_ids:
        # check for missing entries
        if len(answers_dict[id]) == 0:
            print(f'System {id} appears to have no obvious DC power aggregator name.')
        # check for duplicates
        elif len(answers_dict[id]) != 1:
            print(f'System {id} has multiple DC power aggregators!')
            for term in answers_dict[id]:
                print(term)

    return answers_dict

Verify that the manually excluded terms have ac power and dc voltage

In [39]:
for id in {1367, 1432, 1433}:
    relevant_rows_metrics = metrics_df[metrics_df['system_id']==id]
    assert('ac_power' in relevant_rows_metrics.sensor_name.unique())
    assert(any(relevant_rows_metrics.sensor_name.str.contains('dc_voltage')))

In [158]:
dc_agg_names = sort_all_dc_power_names_step_1()

In [160]:
my_units = []
for id in dc_agg_names.keys():
    for metric_dict in dc_agg_names[id]:
        my_units.append(metric_dict['units'])
my_units = set(my_units)
my_units


{'W', 'kW'}

In [64]:
pd.Series(
    {j: False for j in range(10)}
)

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
dtype: bool

In [ ]:
def sort_all_dc_power_names_step_2():
    global metrics_df, systems_cleaned
    # grab the aggregate power names from Step 1
    power_names = sort_all_dc_power_names_step_1()
    # prep my series of booleans for having sub-parts
    has_dc_power_subparts = pd.Series(
        {id: False for id in power_names.keys()}, dtype='bool'
    )
    dc_agg_sensor_names = ['dc_power', 'dc_power_hW', 'dc_power_kW', 'dc_power_1_6',
                           'InvPDC_kW_Avg', 'dc_power_calc']
    for id in power_names.keys():
        # grab only those metrics with both dc and power
        relevant_rows_metrics = metrics_df[metrics_df['system_id']==id]
        dc_and_power_metrics = metrics_search_for_two_fragments_df(
            relevant_rows_metrics,
            'dc',
            'pow',
            'and'
        )
        # drop the aggregate names
        dc_power_non_agg_metrics = dc_and_power_metrics[
            ~dc_and_power_metrics['sensor_name'].isin(dc_agg_sensor_names)
        ]
        #see if any terms remaining
        num_subparts = dc_power_non_agg_metrics.shape[0]
        if num_subparts > 0:
            has_dc_power_subparts[id] = True
            # sort remaining names
            dc_power_non_agg_metrics = dc_power_non_agg_metrics.sort_values('sensor_name')
            # add the partial names on there
            for j in range(0, num_subparts):
                jth_metric = dc_power_non_agg_metrics.iloc[j, :]
                # cleaning step -- some sub-parts had no units!
                jth_unit = jth_metric['units']
                if (jth_unit.lower() != 'w') and (jth_unit.lower() != 'kw'):
                    # should only happen with system 1207
                    assert(id == 1207)
                    # grab the unit from the unit in the aggregate collector
                    # should be 'W'
                    jth_unit = power_names[1207][0]['units']
                power_names[id].append({
                    'metric_id': jth_metric['metric_id'],
                    'sensor_name': jth_metric['sensor_name'],
                    'common_name': jth_metric['common_name'],
                    'units': jth_unit,
                    'calc_details': jth_metric['calc_details'],
                    'whole_or_part': 'part',
                    'index': j + 1
                })
    return (power_names, has_dc_power_subparts)
        


In [167]:
dc_power_names, dc_power_subparts = sort_all_dc_power_names_step_2()

In [168]:
units_collection = []
for id in dc_power_names.keys():
    for metric_dict in dc_power_names[id]:
        units_collection.append(metric_dict['units'])
units_set = set(units_collection)
units_set
        

{'W', 'kW'}

In [169]:
for id in dc_power_names.keys():
    find_nulls_id = []
    for metric_dict in dc_power_names[id]:
        if metric_dict['units'] == '-':
            find_nulls_id.append(metric_dict['sensor_name'])
    if len(find_nulls_id) > 0:
        print(f'System {id} has multiple DC sensors with no units:')
        print(find_nulls_id)

In [96]:
dc_power_names[1283]

[{'metric_id': np.int32(1136),
  'sensor_name': 'dc_power',
  'common_name': 'DC power',
  'units': 'W',
  'calc_details': 'inv1_dc_power+inv2_dc_power',
  'whole_or_part': 'whole'},
 {'metric_id': np.int32(1134),
  'sensor_name': 'inv1_dc_power',
  'common_name': 'DC power',
  'units': 'W',
  'calc_details': 'inv1_dc_voltage*inv1_dc_current',
  'whole_or_part': 'part',
  'index': 1},
 {'metric_id': np.int32(1135),
  'sensor_name': 'inv2_dc_power',
  'common_name': 'DC power',
  'units': 'W',
  'calc_details': 'inv2_dc_voltage*inv2_dc_current',
  'whole_or_part': 'part',
  'index': 2}]

In [70]:
dc_power_subparts

2       False
3       False
1283     True
1284    False
4       False
1416     True
1289    False
10      False
1419    False
1420    False
1422    False
1423    False
1300    False
1429    False
1430    False
1431    False
1305     True
1306     True
1307     True
1308     True
1310     True
1312     True
33      False
34      False
35      False
36      False
4901     True
4902     True
4903     True
1199     True
1200     True
1201    False
1202     True
1203     True
1204     True
1332     True
50       True
1207     True
1208     True
51       True
1229    False
1230    False
1368     True
1369     True
1403    False
1276    False
1277    False
1278     True
dtype: bool

In [191]:
def dc_power_total_name(size_standard: str):
    return f'dc_power_whole_{size_standard}'


def dc_power_partial_name(size_standard: str, ind: int):
    return f'dc_power_subsystem_{ind}_{size_standard}'


def dc_power_dataframe_generator(system_id: int, 
                                 tall_or_wide: str,
                                 error_on_no_data: bool,
                                 size_standard: str):
    '''Make the (tall or wide) pandas DataFrame with all dc power data.'''
    dc_power_names, _ = sort_all_dc_power_names_step_2()
    try:
        my_dc_power_names = dc_power_names[system_id]
    except KeyError:
        if error_on_no_data:
            raise ValueError(f'System {system_id} has no DC power data!')
        else:
            return None
    except BaseException as e:
        raise e
    metric_ids = []
    whole_metric_ids = []
    # grab all metric ids, putting the 'whole' category first
    for metric_data_dict in my_dc_power_names:
        if metric_data_dict['whole_or_part'] == 'whole':
            metric_ids.insert(0, metric_data_dict['metric_id'])
            whole_metric_ids.append(metric_data_dict['metric_id'])
        elif metric_data_dict['whole_or_part'] == 'part':
            metric_ids.append(metric_data_dict['metric_id'])
        else:
            raise ValueError('The "whole_or_part" result of sort_all_power_names_part_2()\n'
                             f'is not correct for system {system_id}.')
    # Load only these metrics from the system
    my_system_parquet_data_path = Path(f'../../../../data_ds_project/systems/parquet/{system_id}/')
    my_system_parquet_selection = pq.ParquetDataset(
        my_system_parquet_data_path, filters=[
            ('metric_id', 'in', metric_ids)
        ]
    )
    system_df = my_system_parquet_selection.read().to_pandas()
    # for reference, 4 columns (see
    # https://github.com/openEDI/documentation/blob/main/pvdaq.md#pvdaq_pvdata)
    # measured_on, utc_measured_on, metric_id, value)
    # standard cleaning
    system_df = system_df.drop_duplicates()
    # See if multiple values at a given time
    # if so, forced to replace value by mean value
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id'])):
        system_df.loc[:, 'mean_value'] = system_df.groupby(
            ['measured_on', 'metric_id']
        )['value'].transform('mean')
        system_df = system_df.drop(columns='value')
        system_df = system_df.rename(columns={'mean_value':'value'})
        system_df.drop_duplicates()
    # if still duplicates, forced to drop utc_measured_on,
    # a frequent source of off-by-one-hour errors
    # (and points with the same 'measured_on' but different 'utc_measured_on'
    # have the same value, so it is likely that utc_measured_on is the problem)
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id', 'value'])):
        system_df = system_df.drop(columns='utc_measured_on')
        system_df = system_df.drop_duplicates()
    # ready to widen the columns
    wide_df = system_df.pivot(
        index='measured_on',
        columns='metric_id',
        values='value'
    )
    # reset the metric_id name of the index of columns
    wide_df.columns.name = ''
    # reset the index
    wide_df = wide_df.reset_index()
    # before continuing, standardize the capitalization of the size term
    if size_standard.lower() == 'w':
        size_standard = 'W'
    elif size_standard.lower() == 'kw':
        size_standard = 'kW'
    else:
        raise ValueError('Only supports watts and kilowatts for now.')
    # standardize units -- probably only necessary for power and temperature
    # Irradiance is pretty clearly in W/m^2, current in A, voltage in V
    if size_standard == 'W':
        for metric_data_dict in my_dc_power_names:
            if metric_data_dict['units'].lower() == 'kw':
                wide_df.loc[:, metric_data_dict['metric_id']] = wide_df[metric_data_dict['metric_id']] * 1000
    elif size_standard == 'kW':
        for metric_data_dict in my_dc_power_names:
            if metric_data_dict['units'].lower() == 'w':
                wide_df.loc[:, metric_data_dict['metric_id']] = wide_df[metric_data_dict['metric_id']] / 1000
    else:
        raise ValueError('Only supports watts and kilowatts for now.')
    # push the 'whole' columns to the beginning of the pack
    # despite re-ordering earlier, can still be loaded in the wrong order.
    reordered_columns = ['measured_on'] + whole_metric_ids+ (wide_df.columns.drop(
        ['measured_on'] + whole_metric_ids
    ).tolist())
    wide_df = wide_df[reordered_columns]
    # rename columns
    renamer_dict = dict()
    for metric_data_dict in my_dc_power_names:
        if metric_data_dict['whole_or_part'] == 'whole':
            renamer_dict[metric_data_dict['metric_id']] = dc_power_total_name(size_standard)
        elif metric_data_dict['whole_or_part'] == 'part':
            renamer_dict[metric_data_dict['metric_id']] = dc_power_partial_name(
                size_standard, metric_data_dict['index']
            )
        else:
            raise ValueError('The "whole_or_part" result of sort_all_power_names_part_2()\n'
                             f'is not correct for system {system_id}.')
    wide_df = wide_df.rename(columns=renamer_dict)
    # convert back to tall format if that is what we wanted
    if tall_or_wide == 'wide':
        return wide_df
    elif tall_or_wide == 'tall':
        return wide_df.melt(
            id_vars='measured_on',
            value_vars=list(renamer_dict.values()),
            var_name='metric_name',
            value_name='value'
        )
    else:
        raise ValueError('The term "tall_or_wide" must be "tall" or "wide.\n'
                         + f'Recieved {tall_or_wide}')


### Test

In [184]:
test_1207 = dc_power_dataframe_generator(1207, 'wide', False, 'kW')

In [185]:
test_1207.head()

,measured_on,dc_power_whole_kW,dc_power_subsystem_1_kW,dc_power_subsystem_2_kW,dc_power_subsystem_3_kW,dc_power_subsystem_4_kW,dc_power_subsystem_5_kW,dc_power_subsystem_6_kW
0,2011-04-22 15:40:00,3.0620,0.5329,0.5208,0.5217,0.5264,0.4953,0.4649
1,2011-04-22 15:50:00,5.0174,0.8660,0.8480,0.8590,0.8700,0.8180,0.7564
2,2011-04-22 16:00:00,3.8773,0.6714,0.6546,0.6653,0.6755,0.6355,0.5750
3,2011-04-22 16:10:00,2.9419,0.5111,0.4940,0.5041,0.5152,0.4817,0.4358
4,2011-04-22 16:20:00,2.2972,0.3989,0.3823,0.3936,0.4043,0.3730,0.3451


In [187]:
test_1207['manual_sum'] = test_1207['dc_power_subsystem_1_kW'] + test_1207['dc_power_subsystem_2_kW'] + test_1207['dc_power_subsystem_3_kW']\
    + test_1207['dc_power_subsystem_4_kW'] + test_1207['dc_power_subsystem_5_kW'] + test_1207['dc_power_subsystem_6_kW']  

In [188]:
test_1207['diff'] = np.isclose(test_1207['dc_power_whole_kW'].values, test_1207['manual_sum'].values)

In [190]:
test_1207['diff'].value_counts()

diff
True    357816
Name: count, dtype: int64

In [149]:
power_agg_names = sort_all_dc_power_names_step_1()
for id in power_agg_names.keys():
    print(id)
    temp_df = dc_power_dataframe_generator(id, 'wide', True, 'W')
    print(temp_df.columns)

2
Index(['measured_on', 'utc_measured_on', 'metric_id', 'value'], dtype='str')
Index(['measured_on', 'dc_power_whole_W'], dtype='str', name='')
3
Index(['measured_on', 'utc_measured_on', 'metric_id', 'value'], dtype='str')
Index(['measured_on', 'dc_power_whole_W'], dtype='str', name='')
1283
Index(['measured_on', 'utc_measured_on', 'metric_id', 'value'], dtype='str')
Index(['measured_on', 'dc_power_subsystem_1_W', 'dc_power_subsystem_2_W',
       'dc_power_whole_W'],
      dtype='str', name='')
1284
Index(['measured_on', 'utc_measured_on', 'metric_id', 'value'], dtype='str')
Index(['measured_on', 'dc_power_whole_W'], dtype='str', name='')
4
Index(['measured_on', 'utc_measured_on', 'metric_id', 'value'], dtype='str')
Index(['measured_on', 'dc_power_whole_W'], dtype='str', name='')
1416
Index(['measured_on', 'utc_measured_on', 'metric_id', 'value'], dtype='str')
Index(['measured_on', 'dc_power_subsystem_2_W', 'dc_power_subsystem_1_W',
       'dc_power_whole_W'],
      dtype='str', name='

In [145]:
dc_power_dataframe_generator(4, 'wide', False, 'kW')

Index(['measured_on', 'utc_measured_on', 'metric_id', 'mean_value'], dtype='str')


KeyError: Index(['value'], dtype='str')

## Testing ground -- Can ignore after this.

In [106]:
path_1283 = Path(f'../../../../data_ds_project/systems/parquet/1283/')
pq_1283 = pq.ParquetDataset(
    path_1283, filters=[
            ('metric_id', 'in', [1136, 1134, 1135])
    ]
)
df_1283 = pq_1283.read().to_pandas()
df_1283 = df_1283.drop_duplicates()
df_1283.tail()

,measured_on,utc_measured_on,metric_id,value
46847841,2022-02-08 18:36:00,2022-02-09 01:36:00,1135,23555.9562
46847842,2022-02-08 18:36:15,2022-02-09 01:36:15,1135,23383.2789
46847843,2022-02-08 18:36:30,2022-02-09 01:36:30,1135,23091.3712
46847844,2022-02-08 18:36:45,2022-02-09 01:36:45,1135,22906.8921
46847845,2022-02-08 18:37:00,2022-02-09 01:37:00,1135,22862.8058


In [110]:
wide_1283 = df_1283.pivot(
    index='measured_on',
    columns='metric_id',
    values='value'
)

In [131]:
wide_1283.head()

,1134,1135,1136
measured_on,,,
2012-01-09 17:00:45,0.0,0.0,0.0
2012-01-09 17:01:00,0.0,0.0,0.0
2012-01-09 17:01:15,0.0,0.0,0.0
2012-01-09 17:01:30,0.0,0.0,0.0
2012-01-09 17:01:45,0.0,0.0,0.0


In [130]:
wide_1283.columns.name = ''

In [111]:
wide_1283.describe()

metric_id,1134,1135,1136
count,1.758535e+07,1.758530e+07,1.161351e+07
mean,1.085359e+05,1.498444e+05,3.729155e+05
std,2.633018e+06,2.706288e+06,6.515954e+06
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,5.488920e+04,5.365824e+04
max,6.398400e+07,6.398400e+07,1.279680e+08


In [ ]:
df_1283.groupby('metric_id').mean()

,measured_on,utc_measured_on,value
metric_id,,,
1134,2017-01-20 02:49:01.558431488,2020-09-16 08:45:12.218384128,108535.929947
1135,2017-01-20 02:44:39.528924416,2020-09-16 08:48:05.502534400,149844.364386
1136,2015-03-20 10:26:00.958238720,NaT,372915.506077


In [107]:
first_overview = df_1283[df_1283['measured_on'] == '2018-09-19 10:31:15']
first_overview

,measured_on,utc_measured_on,metric_id,value
35255273,2018-09-19 10:31:15,NaT,1134,0.0000
35255319,2018-09-19 10:31:15,NaT,1135,140159.7083
